## Netflix Dataset

El dataset se encuentra cargado en un lakehouse de Fabric. 

**Nota:** Para correr el archivo en forma local verificar las librerías de python (pyspark, delta-spark, ipython, pandas)

In [121]:
df = spark.read.format("csv").option("header",True).load("Files/netflix1.csv")
display(df)

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 133, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 9101b391-2b9a-4cf0-8b23-efac6ac34ddd)

### Limpieza de Datos

#### Eliminación de Valores Nulos

Conocer la cantidad de filas del dataframe

In [2]:
print("La cantidad de filas que tiene el df es: ",df.count())

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 4, Finished, Available, Finished)

La cantidad de filas que tiene el df es:  8791


Obtención de información del dataframe.

In [3]:
df.describe().show()

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 5, Finished, Available, Finished)

+-------+----------------+-------------+---------------------------------+------------+--------+----------+------------------+--------------------+--------+------------------+
|summary|         show_id|         type|                            title|    director| country|date_added|      release_year|              rating|duration|         listed_in|
+-------+----------------+-------------+---------------------------------+------------+--------+----------+------------------+--------------------+--------+------------------+
|  count|            8791|         8791|                             8791|        8790|    8790|      8790|              8790|                8790|    8789|              8789|
|   mean|            NULL|         NULL|               1124.7692307692307|        NULL|  1944.0|      NULL|2014.1911480259416|                NULL|    NULL|              NULL|
| stddev|            NULL|         NULL|               1042.1800991068478|        NULL|    NULL|      NULL|  8.794154289

Se puede observar que hay columnas que tienen datos nulos (datos menores a 8791). 
Se recorre el dataframe y se elimina las filas que tienen algún dato vacío con el método 'dropna'

In [122]:
df1=df.dropna()
df1.describe().show()

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 134, Finished, Available, Finished)

+-------+-------+-------+---------------------------------+------------+---------+----------+------------------+------+--------+------------------+
|summary|show_id|   type|                            title|    director|  country|date_added|      release_year|rating|duration|         listed_in|
+-------+-------+-------+---------------------------------+------------+---------+----------+------------------+------+--------+------------------+
|  count|   8789|   8789|                             8789|        8789|     8789|      8789|              8789|  8789|    8789|              8789|
|   mean|   NULL|   NULL|               1124.7692307692307|        NULL|     NULL|      NULL|2014.1911480259416|  NULL|    NULL|              NULL|
| stddev|   NULL|   NULL|               1042.1800991068478|        NULL|     NULL|      NULL|  8.79415428974682|  NULL|    NULL|              NULL|
|    min|     s1|  Movie|                           #Alive| A. L. Vijay|Argentina|  1/1/2008|              1925|

Se guarda este dataframe como tabla delta para posterior análisis.

In [127]:
df1.write.format("delta").mode("overwrite").option("path","Tables/peliculas1").saveAsTable("peliculas1")

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 139, Finished, Available, Finished)

#### Reemplazo de valores incompletos

Los valores incompletos o nulos son los eliminados en el proceso anterior. Por lo tanto, se procede a leerlos para ver cuáles son.

In [5]:
from pyspark.sql import functions as F
df_nulls=df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
df_show=df_nulls.show()

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 7, Finished, Available, Finished)

+-------+----+-----+--------+-------+----------+------------+------+--------+---------+
|show_id|type|title|director|country|date_added|release_year|rating|duration|listed_in|
+-------+----+-----+--------+-------+----------+------------+------+--------+---------+
|      0|   0|    0|       1|      1|         1|           1|     1|       2|        2|
+-------+----+-----+--------+-------+----------+------------+------+--------+---------+



Se puede ver que al menos la última columna tiene dos datos NULL. Por lo tanto mostrando esas dos filas, se observa que se trata sólo de esas filas en las que están contenidas las discrepancias.

In [6]:
df.filter(F.col("listed_in").isNull()).show()

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 8, Finished, Available, Finished)

+----------------+-------------+--------------------+---------+-------+----------+------------+--------------------+--------+---------+
|         show_id|         type|               title| director|country|date_added|release_year|              rating|duration|listed_in|
+----------------+-------------+--------------------+---------+-------+----------+------------+--------------------+--------+---------+
|           s8420|        Movie|The Memphis Belle...|     NULL|   NULL|      NULL|        NULL|                NULL|    NULL|     NULL|
|Flying Fortress"|William Wyler|       United States|3/31/2017|   1944|     TV-PG|      40 min|Classic Movies, D...|    NULL|     NULL|
+----------------+-------------+--------------------+---------+-------+----------+------------+--------------------+--------+---------+



Visualizando cada caso, se realiza el siguiente análisis:
- **1° Fila**: No tiene datos a partir de la columna "director". **Corrección**: se reemplaza las columnas ausentes con los valores que más se repiten (Moda). 
- **2° Fila**: Parece que se debió a un error de carga de los datos en la tabla. **Corrección**: se desplaza las columnas hacia la derecha y se agrega un nuevo campo "show_id" con el último valor numérico de la tabla. Para la columna "type" se agrega arbitrariamente "Movie" puesto que es probable que se trate por la duración del mismo. 

##### Corrección 1° Fila

Creamos una función de Python que busque la moda de cualquier columna seleccionada. 

In [7]:
def calcular_moda(dataf, columna):
  conteo_por_valor = dataf.groupBy(columna).agg(F.count("*").alias("conteo"))
  moda_df = conteo_por_valor.orderBy(F.desc("conteo")).limit(1)
  moda = moda_df.select(columna).first()[0]
  return moda

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 9, Finished, Available, Finished)

In [8]:
moda_director = calcular_moda(df1, "director")
moda_country = calcular_moda(df1, "country")
moda_date_added = calcular_moda(df1, "date_added")
moda_release_year = calcular_moda(df1, "release_year")
moda_rating = calcular_moda(df1, "rating")
moda_duration = calcular_moda(df1, "duration")
moda_listed_in = calcular_moda(df1, "listed_in")

print(f"La moda de la columna 'director' es: {moda_director}")
print(f"La moda de la columna 'country' es: {moda_country}")
print(f"La moda de la columna 'date_added' es: {moda_date_added}")
print(f"La moda de la columna 'moda_release_year' es: {moda_release_year}")
print(f"La moda de la columna 'moda_rating' es: {moda_rating}")
print(f"La moda de la columna 'moda_duration' es: {moda_duration}")
print(f"La moda de la columna 'moda_listed_in' es: {moda_listed_in}")


StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 10, Finished, Available, Finished)

La moda de la columna 'director' es: Not Given
La moda de la columna 'country' es: United States
La moda de la columna 'date_added' es: 1/1/2020
La moda de la columna 'moda_release_year' es: 2018
La moda de la columna 'moda_rating' es: TV-MA
La moda de la columna 'moda_duration' es: 1 Season
La moda de la columna 'moda_listed_in' es: Dramas, International Movies


Se puede ver que para el caso particular de la columna "director" tiene como moda el valor "Not Given". Entonces, veamos cuáles son los directores con más apariciones con el fin de analizar si se trata de alguna situación sistemática o es un dato expurio.

In [10]:
director_count = df.groupBy("director").count().orderBy(F.desc("count")).show(10)

StatementMeta(, 56c9e405-1e54-40f7-95fa-63dc318b2a5c, 12, Finished, Available, Finished)

+--------------------+-----+
|            director|count|
+--------------------+-----+
|           Not Given| 2588|
|       Rajiv Chilaka|   20|
| Alastair Fothergill|   18|
|Raúl Campos, Jan ...|   18|
|        Marcus Raboy|   16|
|         Suhas Kadav|   16|
|           Jay Karas|   14|
| Cathy Garcia-Molina|   13|
|     Youssef Chahine|   12|
|     Martin Scorsese|   12|
+--------------------+-----+
only showing top 10 rows



Se observa un gran porcentaje de la totalidad del dataset con ese dato cargado como director. Visualizamos entonces cuántas categorías hay (Columna "type") para decidir qué se reemplaza en este caso particular.

In [11]:
type_count = df.groupBy("type").count().orderBy(F.desc("count")).show()

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 13, Finished, Available, Finished)

+-------------+-----+
|         type|count|
+-------------+-----+
|        Movie| 6126|
|      TV Show| 2664|
|William Wyler|    1|
+-------------+-----+



Existen sólo dos categorías: "Movie" y "TV Show". El tercer dato que se muestra en la tabla resultante se trata del siguiente caso a corregir. Buscamos entonces cuántos "Movie" hay con el dato de "director" "Not Given".

In [12]:
director_count_1=df.select("director").where((df['type']=='Movie')&(df['director']=='Not Given'))
print(f"Hay {director_count_1.count()} registros de 'Movie' con 'Not Given' cargado en la columna 'director'")

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 14, Finished, Available, Finished)

Hay 173 registros de 'Movie' con 'Not Given' cargado en la columna 'director'


In [13]:
director_count_2=df.select("director").where((df['type']=='TV Show')&(df['director']=='Not Given'))
print(f"Hay {director_count_2.count()} registros de 'TV Show' con 'Not Given' cargado en la columan 'director'")

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 15, Finished, Available, Finished)

Hay 2415 registros de 'TV Show' con 'Not Given' cargado en la columan 'director'


Debido a la gran cantidad de datos sin director conocido e independientemente de si es "Movie" o "TV show", se concluye que el dato a completar en este caso no es representativo y no sería desacertado reemplazarlo con la moda de esta columna también.

In [14]:
reemplazos = {
    "director": moda_director,
    "country": moda_country,
    "date_added": moda_date_added,
    "release_year": moda_release_year,
    "rating": moda_rating,
    "duration": moda_duration,
    "listed_in": moda_listed_in
    }

condicion = F.col("show_id") == "s8420"

for col_name, nuevo_valor in reemplazos.items():
    df = df.withColumn(col_name,F.when(condicion, nuevo_valor).otherwise(F.col(col_name)))
df2=df

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 16, Finished, Available, Finished)

Vericación de lo reemplazado

In [15]:
df_test = df2.where(df2['show_id']=='s8420').show()

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 17, Finished, Available, Finished)

+-------+-----+--------------------+---------+-------------+----------+------------+------+--------+--------------------+
|show_id| type|               title| director|      country|date_added|release_year|rating|duration|           listed_in|
+-------+-----+--------------------+---------+-------------+----------+------------+------+--------+--------------------+
|  s8420|Movie|The Memphis Belle...|Not Given|United States|  1/1/2020|        2018| TV-MA|1 Season|Dramas, Internati...|
+-------+-----+--------------------+---------+-------------+----------+------------+------+--------+--------------------+



In [16]:
df2.summary().show()

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 18, Finished, Available, Finished)

+-------+----------------+-------------+---------------------------------+------------+--------+----------+------------------+--------------------+--------+------------------+
|summary|         show_id|         type|                            title|    director| country|date_added|      release_year|              rating|duration|         listed_in|
+-------+----------------+-------------+---------------------------------+------------+--------+----------+------------------+--------------------+--------+------------------+
|  count|            8791|         8791|                             8791|        8791|    8791|      8791|              8791|                8791|    8790|              8790|
|   mean|            NULL|         NULL|               1124.7692307692307|        NULL|  1944.0|      NULL|2014.1915813424346|                NULL|    NULL|              NULL|
| stddev|            NULL|         NULL|               1042.1800991068478|        NULL|    NULL|      NULL| 8.7937478243

##### Corrección 2° Fila

Para este caso en primer lugar vamos a averiguar cuál es el valor numérico más grande en la columna "show_id". Para ello, utilizamos el dataframe del punto 1 y agregamos otra columna transformándola en variables integer. 

In [30]:
from pyspark.sql.types import *

df3 = df1.withColumn("show_id_number", F.regexp_replace(df1.show_id, "s", ""))
df3 = df3.withColumn("show_id_number", df3["show_id_number"].cast(IntegerType()))

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 32, Finished, Available, Finished)

In [34]:
df3.describe().show()

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 36, Finished, Available, Finished)

+-------+-------+-------+---------------------------------+------------+---------+----------+------------------+------+--------+------------------+------------------+
|summary|show_id|   type|                            title|    director|  country|date_added|      release_year|rating|duration|         listed_in|    show_id_number|
+-------+-------+-------+---------------------------------+------------+---------+----------+------------------+------+--------+------------------+------------------+
|  count|   8789|   8789|                             8789|        8789|     8789|      8789|              8789|  8789|    8789|              8789|              8789|
|   mean|   NULL|   NULL|               1124.7692307692307|        NULL|     NULL|      NULL|2014.1911480259416|  NULL|    NULL|              NULL|4398.9249061326655|
| stddev|   NULL|   NULL|               1042.1800991068478|        NULL|     NULL|      NULL|  8.79415428974682|  NULL|    NULL|              NULL| 2542.358163130376

Como se puede ver el máximo valor es 8807. Por lo tanto se agrega un registro con un valor de "show_id" sumándole una unidad. Se lo agregará al final de la tabla obtenida en el paso anterior ("df2"). 

In [31]:
max_value = df3.agg({"show_id_number": "max"}).collect()[0][0]
show_id_add= "s"+str(max_value+1)

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 33, Finished, Available, Finished)

Asumiendo que este registro se trata de una "Movie" puesto que es parte de la información que hay en una de las celdas, procedemos a crear un dataframe con estos datos para poder incorporarlos en la tabla. Para ello, primero obtenemos en una lista los valores conocidos de la fila en cuestión.

In [32]:
row_corr = df.filter(F.col("show_id") == 'Flying Fortress"').first()
values_corr = row_corr.asDict().values()
list_values=list(values_corr)[:-2]
print(list_values)

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 34, Finished, Available, Finished)

['Flying Fortress"', 'William Wyler', 'United States', '3/31/2017', '1944', 'TV-PG', '40 min', 'Classic Movies, Documentaries']


Agregamos los dos valores ausentes para completar la fila

In [33]:
row_add=[show_id_add,"Movie"]+list_values
print(row_add)

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 35, Finished, Available, Finished)

['s8808', 'Movie', 'Flying Fortress"', 'William Wyler', 'United States', '3/31/2017', '1944', 'TV-PG', '40 min', 'Classic Movies, Documentaries']


Creamos un nuevo dataframe con esta fila única y se la agrega al dataframe "df2"

In [22]:
new_row_df = spark.createDataFrame([tuple(row_add)], schema=df.schema)
df3 = df2.union(new_row_df)

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 24, Finished, Available, Finished)

Verficamos lo agregado

In [23]:
df_test = df3.where(df3['show_id']=='s8808').show()

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 25, Finished, Available, Finished)

+-------+-----+----------------+-------------+-------------+----------+------------+------+--------+--------------------+
|show_id| type|           title|     director|      country|date_added|release_year|rating|duration|           listed_in|
+-------+-----+----------------+-------------+-------------+----------+------------+------+--------+--------------------+
|  s8808|Movie|Flying Fortress"|William Wyler|United States| 3/31/2017|        1944| TV-PG|  40 min|Classic Movies, D...|
+-------+-----+----------------+-------------+-------------+----------+------------+------+--------+--------------------+



Por último queda limpiar la fila donde se encontraban los valores nulos. 

In [117]:
df3=df3.dropna()
df3.describe().show()

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 129, Finished, Available, Finished)

+-------+-------+-------+---------------------------------+------------+---------+----------+------------------+------+--------+------------------+------------------+
|summary|show_id|   type|                            title|    director|  country|date_added|      release_year|rating|duration|         listed_in|    show_id_number|
+-------+-------+-------+---------------------------------+------------+---------+----------+------------------+------+--------+------------------+------------------+
|  count|   8789|   8789|                             8789|        8789|     8789|      8789|              8789|  8789|    8789|              8789|              8789|
|   mean|   NULL|   NULL|               1124.7692307692307|        NULL|     NULL|      NULL|2014.1911480259416|  NULL|    NULL|              NULL|4398.9249061326655|
| stddev|   NULL|   NULL|               1042.1800991068478|        NULL|     NULL|      NULL|  8.79415428974682|  NULL|    NULL|              NULL| 2542.358163130376

Se guarda este dataframe como tabla delta para posterior análisis. Asumiendo una arquitectura "Medallion", consideramos esta capa "silver_layer"

In [125]:
df3.write.format("delta").mode("overwrite").option("path","Tables/movies_silver").saveAsTable("movies_silver")

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 137, Finished, Available, Finished)

#### Preguntas de guía

**¿Cuántos valores nulos encontrás en los datos? ¿Los puedes eliminar?**. En la siguiente celda se observan la cantidad de nulos que hay en el dataset. Fueron eliminadas las filas que la contenían y se guardó como tabla delta "peliculas1".

In [37]:
# Función que castea un dataframe a valores integer. 
def transfor_to_int(dff):
  for col_name in df_nulls.columns:
    dff = dff.withColumn(col_name, F.col(col_name).cast("integer"))
  return dff

df_nulls=transfor_to_int(df_nulls)

StatementMeta(, 3c1a904f-72e2-44f0-bb21-7076e3c99980, 39, Finished, Available, Finished)

In [40]:
total_nulls = df_nulls.select(sum(F.col(c) for c in df_nulls.columns)).collect()[0][0]
print("El total de valores nulos en el DataFrame original son:", total_nulls)

StatementMeta(, 3c1a904f-72e2-44f0-bb21-7076e3c99980, 42, Finished, Available, Finished)

Total de valores nulos en el DataFrame: 9


**¿Cuántos valores incompletos encontrás en los datos? ¿Los puedes reemplazar?** Los valores incompletos son los nulos del paso anterior. Fueron reemplazados según el dataframe "df3".

**¿Podés eliminar columnas que no te aportan información? ¿Cuáles son? ¿Por qué las eliminarías?** Las columnas que no aportan información pueden ser: 
- "rating": Podría eliminarse porque no hay un criterio en términos da valores cuantitativos, por lo que puede ser despreciado. 

In [104]:
df_final=df3['show_id','type','title','director','country','date_added','release_year','duration','listed_in']

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 106, Finished, Available, Finished)

**¿Qué tipo de dato es la columna “release_year”? ¿Lo podes convertir a integer?** La columna "release_year" es de tipo "string" por defecto. Puede ser convertida en "integer" con los métodos propios de pyspark ("types.schema"). 

**La columna “listed_in” contiene diferentes valores separados por coma, ¿Podés crear una columna y quedarte con el primer valor?** La columna "listed_in" puede separarse con los métodos "split" de pyspark. 


Se procede ahora a castear el resto de las columnas para tener una tabla final mucho más limpia para poder ser provista para la analítica de datos.

In [106]:
# Limpieza del string "s" en columna "show_id"
df_final = df_final.withColumn("show_id", F.regexp_replace(df_final.show_id, "s", ""))

# Separación de la última fila manteniendo el primer valor.
df_final = df_final.withColumn("listed_in_category", F.split(F.col("listed_in"), ",").getItem(0))#.withColumn("listed_id", split(col("CustomerName"), "").getItem(1))

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 108, Finished, Available, Finished)

Se procede a modificar el tipo de variable de las columnas para que pyspark pueda analizarlo mejor.

In [108]:
df_final = df_final.withColumn("show_id", df_final["show_id"].cast(IntegerType()))\
    .withColumn("date_added",F.to_date(F.col("date_added"), "M/d/yyyy"))\
    .withColumn("release_year", df_final["release_year"].cast(IntegerType()))

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 110, Finished, Available, Finished)

In [114]:
df_final = df_final.withColumnRenamed("type", "tipo") \
                    .withColumnRenamed("title", "titulo") \
                    .withColumnRenamed("country", "pais") \
                    .withColumnRenamed("date_added", "fecha_incorporacion") \
                    .withColumnRenamed("release_year", "año_lanzamiento") \
                    .withColumnRenamed("duration", "duracion") \
                    .withColumnRenamed("title", "Titulo") \
                    .withColumnRenamed("listed_in", "categorias") \
                    .withColumnRenamed("listed_in_category", "categoria_principal") 

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 116, Finished, Available, Finished)

Por último modificamos el nombre de algunas columnas para mayor entendimiento.

In [102]:
df_final = df_final.withColumn("show_id", df_final["show_id"].cast(IntegerType()))\
    .withColumn("date_added",df_final["date_added"].cast(DateType()))\
    .withColumn("release_year", df_final["release_year"].cast(IntegerType()))

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 104, Finished, Available, Finished)

Finalmente guardamos este dataframe como tabla delta. Asumiendo una arquitectura "Medallion", la consideramos la capa "Gold"

In [126]:
df_final.write.format("delta").mode("overwrite").option("path","Tables/movies_gold").saveAsTable("movies_gold")

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 138, Finished, Available, Finished)

In [129]:
df_final.describe().show()

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 141, Finished, Available, Finished)

+-------+------------------+-------+---------------------------------+------------+---------+-------------------+------------------+--------+------------------+-------------------+
|summary|           show_id|   Tipo|                           Titulo|    director|     pais|fecha_incorporacion|   año_lanzamiento|duracion|        categorias|categoria_principal|
+-------+------------------+-------+---------------------------------+------------+---------+-------------------+------------------+--------+------------------+-------------------+
|  count|              8789|   8789|                             8789|        8789|     8789|               8789|              8789|    8789|              8789|               8789|
|   mean|4398.9249061326655|   NULL|               1124.7692307692307|        NULL|     NULL|               NULL|2014.1911480259416|    NULL|              NULL|               NULL|
| stddev| 2542.358163130376|   NULL|               1042.1800991068478|        NULL|     NULL|  

In [130]:
df_final.summary().show()

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 142, Finished, Available, Finished)

+-------+------------------+-------+---------------------------------+------------+---------+-------------------+------------------+--------+------------------+-------------------+
|summary|           show_id|   Tipo|                           Titulo|    director|     pais|fecha_incorporacion|   año_lanzamiento|duracion|        categorias|categoria_principal|
+-------+------------------+-------+---------------------------------+------------+---------+-------------------+------------------+--------+------------------+-------------------+
|  count|              8789|   8789|                             8789|        8789|     8789|               8789|              8789|    8789|              8789|               8789|
|   mean|4398.9249061326655|   NULL|               1124.7692307692307|        NULL|     NULL|               NULL|2014.1911480259416|    NULL|              NULL|               NULL|
| stddev| 2542.358163130376|   NULL|               1042.1800991068478|        NULL|     NULL|  

In [116]:
display(df_final)

StatementMeta(, d1f33c59-7ee0-4665-8091-734f34b28a37, 121, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 8de5e7be-466c-4e59-9903-ddf093c47fe9)

![image-alt-text](image-URL)